# Task 3: Implement RAG With Unsloth's Dynamic 4-bit Quantization

This notebook builds a **Retrieval-Augmented Generation (RAG)** pipeline using:
- **Unsloth** dynamic 4-bit quantized LLM for memory-efficient generation
- **ChromaDB** as a lightweight vector store for document retrieval
- **Sentence-Transformers** for embedding domain-specific medical documents

The pipeline:
1. Loads a dynamic 4-bit quantized model via Unsloth (`unsloth-bnb-4bit`)
2. Indexes domain-specific medical documents into a vector store
3. Retrieves relevant chunks based on user queries
4. Generates grounded, context-aware responses using the quantized LLM

## 1. Install Required Libraries

In [ ]:
%%capture
%pip install unsloth
%pip install chromadb sentence-transformers langchain langchain-community

## 2. Import Libraries & Check GPU

In [ ]:
import torch
import gc
from unsloth import FastLanguageModel
from transformers import TextStreamer
import chromadb
from sentence_transformers import SentenceTransformer

# Check GPU availability and VRAM
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU: {gpu_name} | VRAM: {vram_gb:.1f} GB")
else:
    raise RuntimeError("No GPU found! Go to Runtime > Change runtime type > GPU (T4).")

## 3. Load Unsloth Dynamic 4-bit Quantized Model

We use an `unsloth-bnb-4bit` variant which applies **dynamic quantization** — selectively preserving full precision for critical parameters (e.g., attention heads, layer norms) while quantizing the bulk of weights to 4-bit. This keeps VRAM usage low (~5-6 GB) while maintaining generation quality.

In [ ]:
# Model configuration
MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct-unsloth-bnb-4bit"
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,           # Auto-detect (float16 on T4)
    load_in_4bit=True,    # Dynamic 4-bit quantization via bitsandbytes
)

# Switch to inference mode for optimized generation
FastLanguageModel.for_inference(model)

print(f"Model loaded: {MODEL_NAME}")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 4. Prepare Domain-Specific Medical Documents

We create a corpus of medical documents covering various topics. In a production setting, these could be loaded from PDFs, clinical notes, or medical databases.

In [ ]:
# Domain-specific medical documents for the knowledge base
medical_documents = [
    {
        "id": "doc_1",
        "title": "Hypertension Overview",
        "text": "Hypertension (high blood pressure) is defined as a systolic blood pressure of 130 mmHg or higher, or a diastolic blood pressure of 80 mmHg or higher. It is a major risk factor for cardiovascular disease, stroke, and kidney failure. Primary (essential) hypertension accounts for 90-95% of cases and has no identifiable cause. Secondary hypertension is caused by an underlying condition such as kidney disease, adrenal gland tumors, or thyroid problems. First-line treatment includes lifestyle modifications: dietary changes (DASH diet), sodium restriction (<2300 mg/day), regular aerobic exercise (150 min/week), weight management, and limiting alcohol intake. Pharmacological treatment includes ACE inhibitors (e.g., lisinopril), ARBs (e.g., losartan), calcium channel blockers (e.g., amlodipine), and thiazide diuretics (e.g., hydrochlorothiazide)."
    },
    {
        "id": "doc_2",
        "title": "Type 2 Diabetes Mellitus",
        "text": "Type 2 Diabetes Mellitus (T2DM) is characterized by insulin resistance and relative insulin deficiency, leading to chronic hyperglycemia. Diagnostic criteria include fasting plasma glucose >= 126 mg/dL, HbA1c >= 6.5%, or 2-hour plasma glucose >= 200 mg/dL during an oral glucose tolerance test. Risk factors include obesity (BMI >= 30), sedentary lifestyle, family history, age > 45, and certain ethnicities. Complications are classified as microvascular (retinopathy, nephropathy, neuropathy) and macrovascular (coronary artery disease, peripheral arterial disease, stroke). First-line pharmacotherapy is metformin, which reduces hepatic glucose production. Second-line agents include SGLT2 inhibitors (empagliflozin), GLP-1 receptor agonists (semaglutide), DPP-4 inhibitors (sitagliptin), and sulfonylureas (glipizide). HbA1c target is generally < 7% for most adults."
    },
    {
        "id": "doc_3",
        "title": "Acute Myocardial Infarction",
        "text": "Acute Myocardial Infarction (AMI) occurs when blood flow to a part of the heart muscle is blocked, usually by a thrombus forming on a ruptured atherosclerotic plaque in a coronary artery. STEMI (ST-elevation MI) involves transmural ischemia and requires emergent reperfusion therapy. NSTEMI (non-ST-elevation MI) involves subendocardial ischemia. Classic symptoms include substernal chest pain radiating to the left arm or jaw, diaphoresis, dyspnea, and nausea. Diagnosis involves ECG changes (ST elevation, new LBBB, or ST depression/T-wave inversions), elevated cardiac biomarkers (troponin I or T), and clinical presentation. Initial management includes aspirin 325 mg chewed, nitroglycerin, morphine for pain, oxygen if SpO2 < 94%, and dual antiplatelet therapy. For STEMI, primary PCI (percutaneous coronary intervention) within 90 minutes is the gold standard; fibrinolytic therapy is an alternative if PCI is unavailable within 120 minutes."
    },
    {
        "id": "doc_4",
        "title": "Community-Acquired Pneumonia",
        "text": "Community-acquired pneumonia (CAP) is an acute infection of the pulmonary parenchyma acquired outside of the hospital setting. The most common causative organism is Streptococcus pneumoniae, followed by Haemophilus influenzae, Mycoplasma pneumoniae (atypical), and respiratory viruses. Symptoms include productive cough, fever, pleuritic chest pain, dyspnea, and signs of consolidation on exam (crackles, bronchial breath sounds, dullness to percussion). Diagnosis is confirmed with chest X-ray showing infiltrates. Severity is assessed using CURB-65 score (Confusion, Urea > 7 mmol/L, Respiratory rate >= 30, Blood pressure < 90/60, age >= 65). Outpatient treatment for healthy adults: amoxicillin 1g TID or doxycycline 100mg BID. For comorbidities: amoxicillin-clavulanate plus a macrolide, or a respiratory fluoroquinolone (levofloxacin or moxifloxacin). Hospitalized patients require IV antibiotics and may need ICU care for severe cases."
    },
    {
        "id": "doc_5",
        "title": "Chronic Kidney Disease",
        "text": "Chronic Kidney Disease (CKD) is defined as abnormalities of kidney structure or function present for more than 3 months, with implications for health. It is staged by GFR: Stage 1 (GFR >= 90, kidney damage with normal GFR), Stage 2 (GFR 60-89, mild decrease), Stage 3a (GFR 45-59), Stage 3b (GFR 30-44), Stage 4 (GFR 15-29, severe decrease), Stage 5 (GFR < 15, kidney failure). The two most common causes are diabetes mellitus and hypertension. Key management includes BP control (target < 130/80 mmHg), ACE inhibitors or ARBs for proteinuria, glycemic control in diabetics, dietary protein restriction (0.8 g/kg/day), phosphorus restriction, and avoidance of nephrotoxins (NSAIDs, contrast dye). Patients with Stage 5 CKD require renal replacement therapy: hemodialysis, peritoneal dialysis, or kidney transplantation. Complications include anemia (treat with EPO), metabolic acidosis, hyperkalemia, and secondary hyperparathyroidism."
    },
    {
        "id": "doc_6",
        "title": "Asthma Management",
        "text": "Asthma is a chronic inflammatory disorder of the airways characterized by variable airflow obstruction, bronchial hyperresponsiveness, and underlying inflammation. Symptoms include recurrent episodes of wheezing, breathlessness, chest tightness, and coughing, particularly at night or early morning. Diagnosis is confirmed by spirometry showing reversible airflow obstruction (FEV1 improvement >= 12% and 200 mL after bronchodilator). Severity classification: intermittent (symptoms <= 2 days/week), mild persistent (> 2 days/week but not daily), moderate persistent (daily symptoms), and severe persistent (symptoms throughout the day). Step-up therapy: Step 1 - SABA as needed; Step 2 - low-dose ICS; Step 3 - low-dose ICS + LABA; Step 4 - medium-dose ICS + LABA; Step 5 - high-dose ICS + LABA + oral corticosteroids or biologics (omalizumab, mepolizumab). Acute exacerbation management: nebulized albuterol, ipratropium bromide, systemic corticosteroids, and supplemental oxygen."
    },
    {
        "id": "doc_7",
        "title": "Deep Vein Thrombosis and Pulmonary Embolism",
        "text": "Deep vein thrombosis (DVT) is the formation of a blood clot in a deep vein, most commonly in the lower extremities. Virchow's triad describes three broad categories contributing to thrombosis: venous stasis, endothelial injury, and hypercoagulability. Risk factors include prolonged immobility, surgery, malignancy, oral contraceptive use, pregnancy, obesity, and inherited thrombophilias (Factor V Leiden, prothrombin gene mutation). Clinical features include unilateral leg swelling, pain, warmth, and erythema. Diagnosis is made using compression ultrasonography and D-dimer testing. Pulmonary embolism (PE) is a life-threatening complication where the thrombus embolizes to the pulmonary vasculature, causing dyspnea, pleuritic chest pain, tachycardia, and hypoxemia. CT pulmonary angiography (CTPA) is the gold standard for PE diagnosis. Treatment for both DVT and PE includes anticoagulation: initial therapy with LMWH (enoxaparin) or direct oral anticoagulants (rivaroxaban, apixaban), followed by at least 3-6 months of anticoagulation. Massive PE may require thrombolysis (alteplase) or surgical embolectomy."
    },
    {
        "id": "doc_8",
        "title": "Major Depressive Disorder",
        "text": "Major Depressive Disorder (MDD) is a mood disorder characterized by persistent depressed mood or loss of interest/pleasure (anhedonia) lasting at least 2 weeks, accompanied by at least 5 of the following symptoms: changes in appetite or weight, insomnia or hypersomnia, psychomotor agitation or retardation, fatigue, feelings of worthlessness or excessive guilt, difficulty concentrating, and recurrent thoughts of death or suicidal ideation. Diagnosis is clinical, based on DSM-5 criteria. First-line pharmacotherapy includes SSRIs (sertraline, escitalopram, fluoxetine) and SNRIs (venlafaxine, duloxetine). These typically require 4-6 weeks for full therapeutic effect. Cognitive Behavioral Therapy (CBT) is the most evidence-based psychotherapy for MDD and is recommended as monotherapy for mild-moderate cases or in combination with medication for moderate-severe cases. Treatment-resistant depression may respond to augmentation strategies (lithium, atypical antipsychotics), ketamine/esketamine, ECT, or TMS."
    },
    {
        "id": "doc_9",
        "title": "Chronic Obstructive Pulmonary Disease",
        "text": "Chronic Obstructive Pulmonary Disease (COPD) is a progressive lung disease characterized by persistent airflow limitation that is not fully reversible. It encompasses chronic bronchitis (productive cough for >= 3 months in 2 consecutive years) and emphysema (destruction of alveolar walls). The primary risk factor is cigarette smoking (responsible for 85-90% of cases). Diagnosis requires spirometry showing post-bronchodilator FEV1/FVC ratio < 0.70. GOLD classification stages severity by FEV1: GOLD 1 (>= 80% predicted, mild), GOLD 2 (50-79%, moderate), GOLD 3 (30-49%, severe), GOLD 4 (< 30%, very severe). Management: smoking cessation is the single most effective intervention. Pharmacotherapy includes short-acting bronchodilators (SABA/SAMA) for rescue, long-acting bronchodilators (LABA: salmeterol, LAMA: tiotropium) for maintenance, and inhaled corticosteroids added for frequent exacerbations. Supplemental oxygen therapy is indicated when PaO2 < 55 mmHg. Pulmonary rehabilitation improves exercise capacity and quality of life."
    },
    {
        "id": "doc_10",
        "title": "Stroke: Ischemic and Hemorrhagic",
        "text": "Stroke is a medical emergency caused by interruption of blood supply to the brain. Ischemic stroke (87% of cases) results from arterial occlusion by thrombosis or embolism, while hemorrhagic stroke (13%) results from rupture of a blood vessel (intracerebral or subarachnoid hemorrhage). The FAST mnemonic helps identify stroke: Face drooping, Arm weakness, Speech difficulty, Time to call emergency services. Rapid assessment includes NIHSS score, non-contrast CT (to rule out hemorrhage), and CT angiography. For ischemic stroke, IV alteplase (tPA) is indicated within 4.5 hours of symptom onset. Mechanical thrombectomy is recommended for large vessel occlusion within 24 hours in selected patients. Secondary prevention includes antiplatelet therapy (aspirin, clopidogrel), statin therapy, BP control, and anticoagulation for atrial fibrillation (warfarin or DOACs). Hemorrhagic stroke management focuses on BP reduction (target SBP < 140 mmHg), reversal of anticoagulation, and potential surgical evacuation for large hematomas."
    }
]

print(f"Loaded {len(medical_documents)} medical documents")
for doc in medical_documents:
    print(f"  - {doc['title']} ({len(doc['text'])} chars)")

## 5. Build the Vector Store with ChromaDB

We chunk the documents and embed them using a lightweight sentence-transformer model. ChromaDB stores the embeddings for fast similarity-based retrieval.

In [ ]:
def chunk_text(text, chunk_size=512, overlap=64):
    """Split text into overlapping chunks by character count."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap
    return chunks

# Prepare chunks with metadata
all_chunks = []
all_ids = []
all_metadatas = []

for doc in medical_documents:
    chunks = chunk_text(doc["text"])
    for i, chunk in enumerate(chunks):
        all_chunks.append(chunk)
        all_ids.append(f"{doc['id']}_chunk_{i}")
        all_metadatas.append({"title": doc["title"], "doc_id": doc["id"], "chunk_index": i})

print(f"Created {len(all_chunks)} chunks from {len(medical_documents)} documents")

In [ ]:
# Load embedding model (lightweight, runs on CPU to save GPU VRAM for the LLM)
embedding_model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")

# Generate embeddings for all chunks
chunk_embeddings = embedding_model.encode(all_chunks, show_progress_bar=True, batch_size=32)
print(f"Embedding dimension: {chunk_embeddings.shape[1]}")

In [ ]:
# Create ChromaDB collection and index the chunks
chroma_client = chromadb.Client()  # In-memory client (lightweight for Colab)

collection = chroma_client.create_collection(
    name="medical_knowledge_base",
    metadata={"hnsw:space": "cosine"}  # Use cosine similarity
)

collection.add(
    ids=all_ids,
    embeddings=chunk_embeddings.tolist(),
    documents=all_chunks,
    metadatas=all_metadatas
)

print(f"Indexed {collection.count()} chunks in ChromaDB")

## 6. Build the Retrieval Logic

The retriever takes a user query, embeds it, and finds the top-k most relevant chunks from the vector store.

In [ ]:
def retrieve_relevant_chunks(query, top_k=3):
    """Retrieve the top-k most relevant document chunks for a given query."""
    query_embedding = embedding_model.encode([query]).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )

    retrieved = []
    for i in range(len(results["documents"][0])):
        retrieved.append({
            "text": results["documents"][0][i],
            "title": results["metadatas"][0][i]["title"],
            "similarity": 1 - results["distances"][0][i]  # cosine similarity
        })

    return retrieved

# Test retrieval
test_query = "What is the treatment for a heart attack?"
results = retrieve_relevant_chunks(test_query)
print(f"Query: {test_query}\n")
for i, r in enumerate(results):
    print(f"  [{i+1}] {r['title']} (similarity: {r['similarity']:.3f})")
    print(f"      {r['text'][:120]}...\n")

## 7. Integrate RAG Pipeline — Retrieval + Generation

The RAG pipeline:
1. **Retrieves** relevant document chunks based on the user's query
2. **Constructs** a prompt with the retrieved context
3. **Generates** a grounded response using the quantized LLM

In [ ]:
def build_rag_prompt(query, retrieved_chunks):
    """Build a prompt with retrieved context for the LLM."""
    context_parts = []
    for i, chunk in enumerate(retrieved_chunks):
        context_parts.append(f"[Source: {chunk['title']}]\n{chunk['text']}")

    context_str = "\n\n".join(context_parts)

    messages = [
        {
            "role": "system",
            "content": (
                "You are a medical expert assistant. Answer the user's question accurately "
                "based ONLY on the provided context. If the context does not contain enough "
                "information, state that clearly. Cite the source documents when possible. "
                "Be concise and clinically precise."
            )
        },
        {
            "role": "user",
            "content": f"Context:\n{context_str}\n\nQuestion: {query}"
        }
    ]

    return messages


def rag_generate(query, top_k=3, max_new_tokens=512, temperature=0.7):
    """Full RAG pipeline: retrieve relevant context, then generate a response."""
    # Step 1: Retrieve
    retrieved_chunks = retrieve_relevant_chunks(query, top_k=top_k)

    print(f"Retrieved {len(retrieved_chunks)} chunks:")
    for i, chunk in enumerate(retrieved_chunks):
        print(f"  [{i+1}] {chunk['title']} (similarity: {chunk['similarity']:.3f})")
    print()

    # Step 2: Build prompt with context
    messages = build_rag_prompt(query, retrieved_chunks)

    # Step 3: Tokenize and generate
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    print(f"Prompt tokens: {inputs.shape[1]}")
    print(f"VRAM in use: {torch.cuda.memory_allocated() / 1e9:.2f} GB\n")
    print("=" * 60)
    print("RESPONSE:")
    print("=" * 60)

    streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

    outputs = model.generate(
        input_ids=inputs,
        streamer=streamer,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=True,
        top_p=0.9,
        repetition_penalty=1.1,
        use_cache=True,
    )

    # Decode the generated text (without the prompt)
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    return response

## 8. Demo: RAG Queries

Let's test the full RAG pipeline with several medical queries.

In [ ]:
# Query 1: Heart Attack Treatment
response_1 = rag_generate("What is the emergency treatment protocol for a STEMI heart attack?")

In [ ]:
# Query 2: Diabetes Management
response_2 = rag_generate("What medications are used as second-line therapy for Type 2 Diabetes?")

In [ ]:
# Query 3: Cross-document retrieval
response_3 = rag_generate("A patient has high blood pressure and declining kidney function. What treatment approach should be considered?")

In [ ]:
# Query 4: Pulmonary Embolism
response_4 = rag_generate("How is pulmonary embolism diagnosed and what is the treatment?")

## 9. Memory Efficiency Analysis

Verify VRAM usage with dynamic 4-bit quantization.

In [ ]:
print("=" * 50)
print("MEMORY EFFICIENCY REPORT")
print("=" * 50)
print(f"Model: {MODEL_NAME}")
print(f"Quantization: Dynamic 4-bit (bitsandbytes NF4)")
print(f"Max sequence length: {MAX_SEQ_LENGTH}")
print(f"Embedding model: all-MiniLM-L6-v2 (CPU)")
print(f"Vector store: ChromaDB (in-memory)")
print(f"Documents indexed: {collection.count()} chunks")
print(f"")
print(f"GPU Memory Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"GPU Memory Reserved:  {torch.cuda.memory_reserved() / 1e9:.2f} GB")
print(f"GPU Memory Free:      {(torch.cuda.get_device_properties(0).total_mem - torch.cuda.memory_reserved()) / 1e9:.2f} GB")
print(f"")
print("Dynamic 4-bit quantization selectively preserves precision for")
print("critical parameters (attention, layer norms) while compressing")
print("feed-forward weights to 4-bit, achieving ~75% VRAM reduction")
print("compared to full fp16 loading.")